# THERMA SEARCH AI · HIT-UAV gerçek arşiv denetimi

## Amaç ve doğrulama durumu
Mevcut **HIT-UAV v1.2.1 ZIP** dosyasını Drive'dan okuyup özgün train/val/test bölümlerini inceler. Veri seti indirilmez, tamamen çıkarılmaz ve model eğitilmez. CPU yeterlidir.

Kullanıcı, Zenodo **7633134** kaydından indirdiği arşivin MD5 doğrulamasını tamamladığını bildirdi. Bu notebook önceki MD5'i yeniden doğruladığını iddia etmez; okunan ZIP üyelerinin CRC'si kontrol edilir. Sürüm ve kayıt ID'si kullanıcı tarafından sağlanan kaynak bilgisidir; lisans burada yeniden sorgulanmaz.

**Bu teslimin geliştirme testleri sentetiktir.** Gerçek ölçümler ancak bu notebook sizin Drive arşivinizle çalışınca üretilecektir. Önceki sayımlar sonuçlara kopyalanmaz; hesaplananlarla karşılaştırılır. Kaynak: [Zenodo 7633134](https://zenodo.org/records/7633134), [makale](https://doi.org/10.1038/s41597-023-02066-6).

## Kurulum
Aşağıdaki hücre public kod deposunu kullanır. Veri setini indirmez. Hafif `requirements-colab.txt` yalnızca Pillow ve Matplotlib içerir; `--no-deps` sayesinde Colab'ın ipykernel/jupyter-client paketleri değiştirilmez. Colab'daki mevcut Matplotlib bağımlılıkları kullanılır.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = 'https://github.com/hilalaltunayy/therma-search-ai.git'
CODE_REF = '8d66b6c9e280ea2dcc388df9fc62201e0e6ee901'  # Sınanan analiz modüllerinin sabit commit'i.
# THERMA_CODE_ROOT yalnızca yerel geliştirici/test kullanımında ayarlanır.
local_code_root = os.environ.get('THERMA_CODE_ROOT')
if local_code_root:
    PROJECT_ROOT = Path(local_code_root).resolve()
else:
    PROJECT_ROOT = Path('/content/therma-search-ai-audit')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', '--quiet', REPOSITORY, str(PROJECT_ROOT)], check=True)
    actual_remote = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual_remote != REPOSITORY:
        raise RuntimeError('Mevcut kod klasörünün kaynağı beklenen public depo değil.')
    if subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Kod klasöründe yerel değişiklik var; üzerine yazılmadı. Yeni çalışma zamanı kullanın.')
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--quiet', '--depth', '1', 'origin', CODE_REF], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
                    '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)

if not (PROJECT_ROOT / 'src/data/hit_uav.py').is_file():
    raise FileNotFoundError('Seçili kod revizyonunda HIT-UAV adaptörü yok.')
sys.path.insert(0, str(PROJECT_ROOT))
CODE_REVISION = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Kullanılan kod commit:', CODE_REVISION)

## Drive ve varsayılan yollar
Drive bağlama isteminde hesabınızı seçin. Dosya yolunuz aşağıdakiyle aynıysa manuel ayar gerekmez. Raporlar her çalıştırmada yeni bir `run_...` klasörüne kaydedilir; önceki sonuçların üzerine yazılmaz.

Tüm görüntüler tek tek okunarak çözülür; Drive I/O nedeniyle birkaç dakika sürebilir. Bellekte aynı anda tek görüntü tutulur. Yalnızca üç seçili kutulu örnek ve özetler kaydedilir; ham görüntüler Drive'a tekrar kopyalanmaz.

In [ ]:
# Yerel doğrulamada bu iki ortam değişkeni geçici sentetik ZIP/çıktı yollarını sağlar.
if not local_code_root:
    from google.colab import drive
    drive.mount('/content/drive')

ZIP_PATH = Path(os.environ.get('THERMA_ZIP_PATH',
    '/content/drive/MyDrive/ThermaSearchAI/datasets/hit_uav_v1.2.1.zip'))
REPORT_ROOT = Path(os.environ.get('THERMA_REPORT_ROOT',
    '/content/drive/MyDrive/ThermaSearchAI/reports/hit_uav/'))
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f'Mevcut arşiv bulunamadı; indirme yapılmayacak: {ZIP_PATH}')
VERIFY_IMAGES = True
SAMPLES_PER_SPLIT = 1  # Toplam en fazla 3 örnek; yardımcı fonksiyon split başına en fazla 3 kabul eder.
print('Arşiv:', ZIP_PATH, 'Boyut:', ZIP_PATH.stat().st_size, 'bayt')
print('Kalıcı rapor kökü:', REPORT_ROOT)

## Şema, sayımlar ve kalite denetimi
Adaptör kullanıcı tarafından Colab'da doğrulanan `images[].filename`, `annotation[]`, `image_id`, `category_id` ve piksel `[xmin,ymin,width,height]` biçimini kullanır. Sınıflar: **Person=0, Car=1, Bicycle=2, OtherVehicle=3, DontCare=4**. Görüntü kayıtlarında boyut yoksa bildirilen 640×512 kullanılır; bu durum raporda sayılır. Görüntülerin gerçek çözünürlüğü ayrıca okunur.

Geçersiz kutular, bozuk/eksik görüntüler, kopuk referanslar, tekrarlanan kayıtlar ve beklenmeyen sınıflar raporlanır. Kaynaklar düzeltilmez. **Ham nesne sayıları** tüm annotation satırlarıdır; **kabul edilen kutular** geçerli referans ve sonlu/pozitif koordinatları olanlardır, sınır dışı kutular ayrıca raporlanarak korunur. DontCare sayımlarda tutulur.

Arşivin başındaki sürüm klasörü otomatik bulunur. Güvensiz yollar, sembolik bağlantılar, yinelenen üye yolları ve birden fazla `normal_json` kökü reddedilir. JSON okuma/şema hatasında tahmini sonuçlarla devam edilmez.

In [ ]:
import json
from src.data.hit_uav import CLASSES, SPLITS
from src.data.hit_uav_archive import audit_hit_uav_zip
from src.data.hit_uav_reports import save_audit_reports

print('ZIP üzerinden tüm splitler inceleniyor...')
REPORT, LOADED = audit_hit_uav_zip(ZIP_PATH, verify_images=VERIFY_IMAGES)
print('Bulunan kök:', REPORT['normal_json_root'])
for split, item in REPORT['splits'].items():
    print(split, '| görüntü:', item['image_files'], '| Person:', item['objects_by_class_raw']['Person'],
          '| ham nesne:', item['raw_annotation_records'], '| DontCare:', item['objects_by_class_raw']['DontCare'])
    print('  Çözünürlükler:', item['resolutions'], '| Sorunlar:', item['issue_counts'])

## Kalıcı çıktıları önce kaydet
CSV ve kalite JSON'u grafiklerden önce yazılır. Her çalıştırmanın `analysis_metadata.json` dosyası kod commit'ini, zamanı (UTC), sürümü, kaynak kaydını, ortamı, yapılandırmayı ve üretilen dosyaların SHA256 özetlerini saklar. Çizim yarıda kalırsa durum `partial` olarak kalır. `complete`, veri kusursuz demek değildir; kalite sorunları ayrı JSON içindedir.

Üretilenler: `class_distribution.png`, `person_box_sizes.png`, en fazla üç `sample_*.png`, `split_statistics.csv`, `data_quality.json`, `analysis_metadata.json`. Görüntü dosya adları raporlarda yer alabilir; paylaşmadan önce inceleyin.

In [ ]:
RUN_DIR = save_audit_reports(REPORT, LOADED, ZIP_PATH, REPORT_ROOT,
                              code_revision=CODE_REVISION, samples_per_split=SAMPLES_PER_SPLIT,
                              prior_md5_verified_by_user=True)
print('Raporlar Drive’a kaydedildi:', RUN_DIR)
for path in sorted(RUN_DIR.iterdir()):
    print(path.name, path.stat().st_size, 'bayt')

## Sayım tablosu ve önceki sonuçlarla karşılaştırma
Önceki kullanıcı ölçümleri: train **2029 / 8533 / 17628**, val **290 / 1168 / 2460**, test **579 / 2611 / 4811** (görüntü / Person / tüm nesneler). Bunlar yalnızca karşılaştırma referansıdır. Uyuşmazlıkta yeni sayıların üzerine yazılmaz; sorun araştırılır. Görüntü dosyası sayısıyla JSON görüntü kaydı sayısı ayrıca karşılaştırılmalıdır.

In [ ]:
import csv
from IPython.display import display, Image as DisplayImage
rows = list(csv.DictReader((RUN_DIR / 'split_statistics.csv').open(encoding='utf-8')))
# Pandas gerektirmeden okunabilir tablo.
from IPython.display import HTML
from html import escape
columns = ['split', 'image_files', 'raw_image_records', 'raw_annotation_records', 'Person', 'DontCare', 'quality_issue_count']
display(HTML('<table><tr>' + ''.join('<th>' + c + '</th>' for c in columns) + '</tr>' +
             ''.join('<tr>' + ''.join('<td>' + escape(row[c]) + '</td>' for c in columns) + '</tr>' for row in rows) + '</table>'))
for split, item in REPORT['splits'].items():
    print(split, item['previous_count_comparison'])
    if not all(value['matches'] for value in item['previous_count_comparison'].values()):
        print('UYUŞMAZLIK: önceki kullanıcı ölçümü ile yeniden hesaplanan sayı farklı; rapor kaydedildi.')

## Sınıflar ve küçük insan kutuları
İlk grafik ham annotation satırlarını sınıflara ve özgün splitlere göre gösterir; DontCare ayrı sütundur. İkinci grafik Person genişlik/yükseklik dağılımlarıdır.

Boyut hesabı yalnızca referansı geçerli, pozitif/sonlu ve **etiket boyutuna göre sınır içinde** kalan Person kutularını kapsar; görüntü dosyası eksik/bozuk olabilir. Medyan, kutuların ortadaki boyutunu özetler. `<16` oranının paydası bu kapsamdaki Person kutularıdır; en az bir kenarı 16 pikselden küçük olanların oranını ölçer. Bu COCO küçük nesne metriği veya model performansı değildir. Kullanıcının önceki yaklaşık **12×19** medyanı hesaplamaya girdi olarak verilmez.

Test splitinde bu işlem veri bütünlüğü/keşif içindir; örnekler model veya eşik seçiminde kullanılmamalıdır.

In [ ]:
display(DisplayImage(filename=str(RUN_DIR / 'class_distribution.png')))
display(DisplayImage(filename=str(RUN_DIR / 'person_box_sizes.png')))
for split, item in REPORT['splits'].items():
    print(split, item['person_box_sizes'])
print('Tüm splitler:', REPORT['person_box_sizes_all_splits'])

## Seçili termal örnekler
Her splitte adı sıralanmış ilk okunabilir Person içeren görüntü seçilir; temsil gücü iddia edilmez. Person kutuları ve varsa DontCare bölgeleri (kesik turuncu) çizilir. Diğer sınıflar bu görsellerde gösterilmez; sayımlarda korunur. Etiketler kırpılmaz veya yeniden eşlenmez.

In [ ]:
for path in sorted(RUN_DIR.glob('sample_*.png')):
    display(DisplayImage(filename=str(path)))
metadata = json.loads((RUN_DIR / 'analysis_metadata.json').read_text(encoding='utf-8'))
print('Çıktı durumu:', metadata['status'], '| Örnek çizim hataları:', metadata['sample_errors'][:5])

## Kalite ve bölümler arası ortak dosya adları
Kontrol hem ZIP görüntü dosyalarının hem de kabul edilen etiket görüntü kayıtlarının temel dosya adlarını karşılaştırır. Aynı ad aynı içerik demek değildir. **Ortak ad bulunmaması; yakın kare, aynı uçuş/video/konum veya yeniden kodlanmış görüntü sızıntısı olmadığını kanıtlamaz.** Bu denetim içerik hash'i veya görsel benzerlik incelemesi yapmaz. Splitler değiştirilmez.

Geçerli negatif örnek ile eksik etiket görsel incelemeyle ayrılmalıdır. Kaynak görüntüdeki anotasyonsuz insanları bu araç otomatik keşfedemez. DontCare eğitim/değerlendirme politikası bu notebook tarafından belirlenmez.

In [ ]:
for split, item in REPORT['splits'].items():
    print(split, 'Sorun türleri:', item['issue_counts'])
    for key in ('missing_images', 'unannotated_images', 'empty_annotation_records'):
        print(' ', key, len(item[key]), 'ilk 5:', item[key][:5])
    print(' İlk 5 sorun:', item['issues'][:5])
for pair, item in REPORT['cross_split_filename_overlap'].items():
    print(pair, {key: {'count': len(value), 'first_10': value[:10]} for key, value in item.items()})
print('Tam ayrıntılar:', RUN_DIR / 'data_quality.json')
print(*REPORT['limitations'], sep='\n- ')

## Sonraki değerlendirme
Drive'daki CSV ve JSON'u önceki sayımlarla birlikte inceleyin. Uyuşmazlıklar, bozuk/eksik dosyalar ve sınır dışı kutular için kaynak kaydını kontrol edin; bu notebook düzeltme veya split değişikliği yapmaz. Küçük insan kutularına ilişkin ölçümleri kaydedip gelecekteki eğitim/değerlendirme planında kullanıcı onayına sunun. Bu çalıştırma model eğitimi başlatmaz.


## 9 · Etiketsiz görüntü ve sızıntı denetimi — yapılandırma
Bu bölüm mevcut analizi değiştirmez; aynı ZIP'i salt okunur olarak yeniden tarar. Yeni çıktılar aynı Drive rapor kökü altında **ayrı ve benzersiz `run_leakage_.../`** klasörüne kaydedilir. Yeni bölüm tekrar çalıştırıldığında önceki raporlar ve manuel kararlar korunur.

Kullanıcının gerçek Colab incelemesi: sıfır anotasyonlu train **21**, val **3**, test **8**, toplam **32** görüntü. Bu sayılar yeniden hesaplanır; farklı sonuçta rapor uyarır, sayıyı zorla 32 yapmaz. Seçim ham `annotation` referanslarına dayanır: DontCare da anotasyondur. Hatalı kutular yüzünden boş kalan kayıtlar ve kayıp görüntü kayıtları ayrıca işaretlenir; sıfır ham anotasyonla birleştirilmez.

**Hiçbir görüntü otomatik negatif kabul edilmez.** CSV'deki `human_present`, `reviewer`, `review_notes` alanları boş bırakılır. `human_present` için `yes`, `no` veya `uncertain` yazabilirsiniz; bu alanlar etiketi değiştirmez. Contact sheet başlıklarında tam göreli dosya adı ve CSV ile eşleşen `review_id` vardır.

Benzerlik ayarları:
- `dhash_threshold=4`: 64 bit dHash arasında **Hamming ≤4** farkı aday sayar; 0 da piksel eşitliği anlamına gelmez. Tüm okunabilir split-dışı çiftler taranır; aynı split içindeki çiftler kapsam dışıdır.
- Dosya SHA256 eşitliği **aynı dosya baytlarını**, ayrıca hesaplanan piksel SHA256 eşitliği **aynı doğal mod/boyut/piksel verisini** doğrular. Bunlar kesin kopya örtüşmesidir; yakın görüntü adaylarından ayrı raporlanır.
- dHash gri 9×8 LANCZOS küçültmede yatay farklarla hesaplanır. Görsel benzerlik tek başına sızıntı kanıtı değildir; tekdüze termal sahneler yanlış eşleşebilir. 0/tüm-1 hash düşük bilgi uyarısı taşır. Kırpma, döndürme, ton dönüşümü veya aynı uçuşun yakın kareleri kaçabilir; uçuş/konum sızıntısı dışlanmış sayılmaz.
- CSV boyutları sınırlıdır: near adaylarında en fazla 1000, exact çiftlerde 5000 satır. Toplam bulunan çift sayısı ve kesilme bilgisi JSON'da yer alır; **tüm exact grup üyeleri** ayrı JSON'da korunur. Limitleri artırabilirsiniz. Near sırası önce düşük bilgi olmayan hash'ler, sonra en düşük bit farkıdır.
- Adlandırma regex'i yalnızca son sayısal token öncesini bir **grup hipotezi** olarak araştırır; doğrulanmış uçuş kimliği olarak sunmaz. Eşleşmeyen adlar `not_inferred` kalır.


In [ ]:
from src.data.leakage import ReviewConfig
from src.data.leakage_reports import run_leakage_review

LEAKAGE_CONFIG = ReviewConfig(
    dhash_threshold=4,
    max_near_pairs=1000,
    max_exact_pairs=5000,
    max_pair_previews=12,   # En fazla 12 yan yana karşılaştırma; tüm görüntüler kaydedilmez.
    contact_page_size=20,
    group_pattern=r'^(?P<group>.+?)[_-](?P<frame>\d+)$',
)
print(LEAKAGE_CONFIG)

## 10 · ZIP'i tara ve kalıcı inceleme paketini kaydet
CPU ile her görüntü tek tek okunur; Drive I/O zaman alabilir. Dosya ve piksel SHA256'ları, dHash'ler, boş anotasyon seçimi ve olası dosya adı grupları yeniden hesaplanır. Kaynak görüntüler/etiketler/splitler değiştirilmez. Önceki raporların üzerine yazılmaz.

Çıktılar: `contact_sheet_*.png`, `manual_review.csv`, `hash_inventory.csv`, `exact_duplicate_pairs.csv`, `exact_duplicate_groups.json`, `near_duplicate_candidates.csv`, `suspicious_pair_*.png`, `filename_group_hypotheses.csv`, `leakage_summary.json`. Başarısız tamamlanan çalışmanın özeti `partial` kalır; tamamlanmış raporun veri okuma hataları varsa `complete_with_errors` olur.


In [ ]:
print('Sıfır anotasyon, SHA256 ve dHash denetimi başlıyor...')
REVIEW_RUN_DIR, LEAKAGE_SUMMARY = run_leakage_review(
    ZIP_PATH, REPORT_ROOT, code_revision=CODE_REVISION,
    config=LEAKAGE_CONFIG, progress=print,
)
print('Yeni inceleme klasörü:', REVIEW_RUN_DIR)
print('Durum:', LEAKAGE_SUMMARY['status'])
print('Sıfır ham anotasyon:', LEAKAGE_SUMMARY['zero_raw_annotation_counts'])
print('Manuel inceleme nedenleri:', LEAKAGE_SUMMARY['review_selection_reasons'])
print('Önceki 21/3/8 ile karşılaştırma:', LEAKAGE_SUMMARY['previous_user_empty_counts_comparison'])
print('Exact ve near çift özeti:', LEAKAGE_SUMMARY['comparison'])
if any(not v['matches'] for v in LEAKAGE_SUMMARY['previous_user_empty_counts_comparison'].values()):
    print('UYARI: sıfır anotasyon sayısı önceki kullanıcı ölçümüyle farklı; JSON ayrıntılarını inceleyin.')
if LEAKAGE_SUMMARY['comparison']['near_csv_truncated'] or LEAKAGE_SUMMARY['comparison']['exact_csv_truncated']:
    print('UYARI: çift CSV limiti doldu. Toplam sayılar özette; tüm exact gruplar JSON içinde.')
print('Adlandırmadan çıkarılamayan görüntü:', LEAKAGE_SUMMARY['filename_group_not_inferred'])
print('Splitler arasında ortak grup hipotezleri (ilk 5):',
      LEAKAGE_SUMMARY['cross_split_filename_group_hypotheses'][:5])
print('Hatalar (ilk 5):', LEAKAGE_SUMMARY['errors'][:5])

## 11 · Contact sheet ve şüpheli çiftleri görsel olarak incele
Aşağıda her splitin tüm contact sheet sayfaları ve her kategoriden en fazla iki çift gösterilir. Tüm sayfalar Drive'a da kaydedilir; `leakage_summary.json` içindeki `image_manifest` hangi görüntünün hangi sayfada olduğunu gösterir. İlgili CSV'leri Drive'da açıp inceleyin; script daha önceki manuel kararları okumaz veya etiketlere uygulamaz.

`confirmed_identical_file_bytes` / `confirmed_identical_decoded_pixels` splitler arası tam kopya örtüşmesidir. `review_candidate` ise yalnızca benzerlik adayıdır. Aynı adlandırma grubuna düşen dosyalar gerçek uçuş grubu olarak kabul edilmez. Test örnekleri burada bütünlük denetimi içindir; model/eşik seçimi için kullanılmamalıdır.


In [ ]:
for split in SPLITS:
    pages = sorted(REVIEW_RUN_DIR.glob(f'contact_sheet_{split}_*.png'))
    if pages:
        for page in pages:
            display(DisplayImage(filename=str(page)))
        print(split, 'contact sheet sayfası:', len(pages), '| tamamı Drive’da')
    else:
        print(split, 'Manuel inceleme için seçilen mevcut görüntü yok.')
for category in ('exact', 'near'):
    entries = [row for row in LEAKAGE_SUMMARY['image_manifest']
               if 'classification' in row and
               ((row['classification'] == 'review_candidate') == (category == 'near'))]
    for row in entries[:2]:
        display(DisplayImage(filename=str(REVIEW_RUN_DIR / row['output'])))
print('Manuel karar tablosu:', REVIEW_RUN_DIR / 'manual_review.csv')
print('Kesin kopya grupları:', REVIEW_RUN_DIR / 'exact_duplicate_groups.json')
print('Benzerlik adayları:', REVIEW_RUN_DIR / 'near_duplicate_candidates.csv')
print('Tüm sınırlar:', *LEAKAGE_SUMMARY['limitations'], sep='\n- ')